In [ ]:
from __future__ import annotations

import argparse
import json
import os
import tempfile
from dataclasses import asdict, dataclass, field
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pywt
from skimage.color import rgb2gray
from skimage import exposure
from skimage.feature import graycomatrix, graycoprops
from skimage.io import imread
from skimage.transform import resize
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


LABELS = {"good": 0, "not-good": 1}
CLASS_NAMES = ["good", "not-good"]

# SteelBlastQC Dataset
The SteelBlastQC dataset contains steel surfaces that are either shot-blasted or still need shot-blasting to achieve the required texture. The dataset includes “good” images (ready for paint) images and 766 “not-good” images (needs shot-blasting) images. As declared by the collaborating manufacturer, the ideally treated surface is clean and uniformly coarse with an average roughness level of SA 2.5. The “not-good” class presents several types of defects to the surface, located by industrial shot-blasting experts. These include: fresh welding lines and cuts, abrasion, corrosion, and discoloration.

# Load Dataset
This step reads training and test images from the structured dataset folders, collects file paths, and generates numeric labels for each image class.
It does not load image arrays immediately; instead it creates a lightweight image path list that is later used for preprocessing, normalization, feature extraction, and classification.


In [ ]:
@dataclass(frozen=True)
class AppConfig:
    dataset_dir: Path = Path("doi-10.34894-ekznn0(1)/SteelBlastQC")
    output_model: Path = Path("steelblast_svm_glcm_dwt.joblib")
    metrics_json: Path = Path("steelblast_svm_glcm_dwt_metrics.json")
    quick_limit: int | None = None

# load either train or test images 
def load_split(dataset_dir: Path, split: str) -> tuple[list[Path], np.ndarray]:
    image_paths: list[Path] = []
    labels: list[int] = []

    for class_name, label in LABELS.items():
        class_dir = dataset_dir / split / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Missing expected folder: {class_dir}")

        paths = sorted(class_dir.glob("*.png"))
        image_paths.extend(paths)
        labels.extend([label] * len(paths))

    if not image_paths:
        raise FileNotFoundError(f"No .png images found in {dataset_dir / split}")

    return image_paths, np.asarray(labels, dtype=np.int64)

    

app_config = AppConfig()


train_paths, y_train = load_split(app_config.dataset_dir, "train")
test_paths, y_test = load_split(app_config.dataset_dir, "test")

# `GLCM` + `DWT` feature pipeline
Handcrafted texture features are data-efficient and interpretable for small/industrial image sets; combining co-occurrence (GLCM) and multi-scale decomposition (DWT) captures complementary texture information while avoiding deep-learning data/compute demands.
- **Input & Resize**: `image_size = 256` — preserves texture detail while bounding computational cost and standardizing feature dimensions for GLCM/DWT.

- **Color & Illumination**: grayscale conversion with `illumination_normalization = "clahe"` (`clahe_clip_limit = 0.01`) and percentile fallback `(2.0, 98.0)`. CLAHE reduces lighting variation yet preserves local texture; percentile clipping provides robust global contrast control.

- **GLCM Design**:
    - `glcm_levels = 32` — limits matrix size and noise sensitivity while retaining characteristic texture gradients.
    - `glcm_distances = (1,2,4,8)` and `glcm_angles = (0, π/4, π/2, 3π/4)` — multi-scale, multi-orientation sampling captures varied blast textures.
    - `glcm_properties = ("contrast","dissimilarity","homogeneity","energy","correlation","ASM")` — summarize local variation, uniformity, and correlation.

- **DWT Design**:
    - `dwt_wavelet = "db2"`, `dwt_level = 3` — Daubechies wavelets provide compact time-frequency localization; three levels capture coarse-to-fine texture without exploding feature count.
    - `describe_coefficients(...)` uses mean, std, absolute moments, energy, entropy, and percentiles to robustly summarize coefficient distributions.

- **Feature Composition & Preprocessing**: concatenate GLCM + DWT features; `StandardScaler` standardizes features because SVMs depend on distances. `quantize_image(...)` stabilizes GLCM inputs.


In [ ]:
@dataclass(frozen=True)
class FeatureExtractionConfig:
    image_size: int = 256 # Resize images to image-size x image-size before feature extraction. This standardization is important for consistent feature extraction, especially for GLCM and DWT, which can be sensitive to image dimensions. A size of 256x256 provides a good balance between retaining detail and keeping computational requirements manageable.
    illumination_normalization: str = "clahe" # Normalize each grayscale image before feature extraction. Gentle CLAHE improves robustness to lighting variation while preserving local blast texture.
    clahe_clip_limit: float = 0.01
    contrast_percentiles: tuple[float, float] = (2.0, 98.0)
    
    glcm_levels: int = 32 # The number of gray levels to quantize the image into for GLCM calculation. Reducing to 32 levels helps manage the size of the co-occurrence matrix and focuses on broader texture patterns, which can be beneficial for classification while keeping computational complexity reasonable. TODO validate that with paper
    glcm_distances: tuple[int, ...] = (1, 2, 4, 8)
    glcm_angles: tuple[float, ...] = (0.0, np.pi / 4, np.pi / 2, 3 * np.pi / 4)
    glcm_properties: tuple[str, ...] = (
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation",
        "ASM",
    )
    dwt_wavelet: str = "db2" #?
    dwt_level: int = 3




# load either train or test images 
def load_split(dataset_dir: Path, split: str) -> tuple[list[Path], np.ndarray]:
    image_paths: list[Path] = []
    labels: list[int] = []

    for class_name, label in LABELS.items():
        class_dir = dataset_dir / split / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Missing expected folder: {class_dir}")

        paths = sorted(class_dir.glob("*.png"))
        image_paths.extend(paths)
        labels.extend([label] * len(paths))

    if not image_paths:
        raise FileNotFoundError(f"No .png images found in {dataset_dir / split}")

    return image_paths, np.asarray(labels, dtype=np.int64)

def normalize_illumination(image: np.ndarray, config: FeatureExtractionConfig) -> np.ndarray:
    image = np.clip(image.astype(np.float32), 0.0, 1.0)
    method = config.illumination_normalization

    if method == "none":
        return image

    if method == "clahe":
        return exposure.equalize_adapthist(
            image,
            clip_limit=config.clahe_clip_limit,
        ).astype(np.float32)

    if method == "percentile":
        lower, upper = np.percentile(image, config.contrast_percentiles)
        if upper <= lower + 1e-6:
            return image
        return np.clip((image - lower) / (upper - lower), 0.0, 1.0).astype(np.float32)

    raise ValueError(
        "--illumination-normalization must be one of: none, clahe, percentile."
    )

# Loops through all images
# Extracts features
# Stacks them into matrix:
def build_feature_matrix(
    image_paths: list[Path],
    config: FeatureExtractionConfig,
    split_name: str,
) -> np.ndarray:
    features = []

    for index, image_path in enumerate(image_paths, start=1):
        features.append(extract_features(image_path, config))
        if index % 100 == 0 or index == len(image_paths):
            print(f"{split_name}: extracted {index}/{len(image_paths)} images")

    return np.vstack(features)

def balanced_limit(
    image_paths: list[Path],
    labels: np.ndarray,
    limit_per_class: int,
) -> tuple[list[Path], np.ndarray]:
    selected_paths: list[Path] = []
    selected_labels: list[int] = []

    for label in sorted(np.unique(labels)):
        class_indices = np.flatnonzero(labels == label)[:limit_per_class]
        selected_paths.extend(image_paths[index] for index in class_indices)
        selected_labels.extend(labels[index] for index in class_indices)

    return selected_paths, np.asarray(selected_labels, dtype=np.int64)

def extract_features(image_path: Path, config: FeatureExtractionConfig) -> np.ndarray:
    image = read_grayscale_image(image_path, config.image_size, config)
    return extract_features_from_image(image, config)

def read_grayscale_image(
    image_path: Path,
    image_size: int,
    config: FeatureExtractionConfig | None = None,
) -> np.ndarray:
    image = imread(image_path)

    if image.ndim == 3:
        image = image[..., :3]
        image = rgb2gray(image) # Convert to grayscale using luminosity method, which accounts for human perception of color brightness. This is important for texture analysis with GLCM, as it relies on intensity values. The resulting grayscale image will have values in the range [0, 1], which is suitable for further processing and feature extraction.

    image = resize(
        image,
        (image_size, image_size),
        anti_aliasing=True,
        preserve_range=True,
    )

    image = image.astype(np.float32)
    if image.max() > 1.0:
        image /= 255.0

    if config is None:
        return image

    return normalize_illumination(image, config)


# get Final feature vector
def extract_features_from_image(image: np.ndarray, config: FeatureExtractionConfig) -> np.ndarray:
    glcm_features = extract_glcm_features(image, config)
    dwt_features = extract_dwt_features(image, config)
    return np.concatenate([glcm_features, dwt_features])

def extract_glcm_features(image: np.ndarray, config: FeatureExtractionConfig) -> np.ndarray:
    #reduce to 32 intensity levels
    quantized = quantize_image(image, config.glcm_levels)
    # Co-occurrence Matrix
    glcm = graycomatrix(
        quantized,
        distances=config.glcm_distances,
        angles=config.glcm_angles,
        levels=config.glcm_levels,
        symmetric=True,
        normed=True,
    )

    features: list[float] = []
    # extract 6 configured features
    for prop in config.glcm_properties:
        values = graycoprops(glcm, prop).ravel()
        features.extend(values)
        features.append(float(values.mean()))
        features.append(float(values.std()))

    return np.asarray(features, dtype=np.float32)

def extract_dwt_features(image: np.ndarray, config: FeatureExtractionConfig) -> np.ndarray:
    # Apply wavelet decomposition
    coeffs = pywt.wavedec2(
        image,
        wavelet=config.dwt_wavelet,
        level=config.dwt_level,
        mode="symmetric",
    )

    features: list[float] = []
    approximation = coeffs[0]
    features.extend(describe_coefficients(approximation))

    # Repeat across 3 levels
    for horizontal, vertical, diagonal in coeffs[1:]:
        features.extend(describe_coefficients(horizontal))
        features.extend(describe_coefficients(vertical))
        features.extend(describe_coefficients(diagonal))

    return np.asarray(features, dtype=np.float32)

def quantize_image(image: np.ndarray, levels: int) -> np.ndarray:
    clipped = np.clip(image, 0.0, 1.0)
    quantized = np.floor(clipped * (levels - 1)).astype(np.uint8)
    return quantized

# DWT coefficients can have a wide range of values, including negative and positive numbers. To capture the texture information effectively, we compute several statistics on the coefficients, such as mean, standard deviation, energy (mean of squared values), and entropy (which measures the randomness or complexity of the coefficients). Additionally, we include percentiles to capture the distribution of coefficient values. These features help summarize the texture information contained in the DWT coefficients in a way that is useful for classification.
def describe_coefficients(coefficients: np.ndarray) -> list[float]:
    values = coefficients.ravel().astype(np.float64)
    abs_values = np.abs(values)
    energy = np.mean(values**2)
    histogram, _ = np.histogram(abs_values, bins=32, density=False)
    probabilities = histogram.astype(np.float64) / max(histogram.sum(), 1)
    probabilities = probabilities[probabilities > 0]
    entropy = -np.sum(probabilities * np.log2(probabilities))

    return [
        float(values.mean()),
        float(values.std()),
        float(abs_values.mean()),
        float(abs_values.std()),
        float(energy),
        float(entropy),
        float(np.percentile(values, 10)),
        float(np.percentile(values, 50)),
        float(np.percentile(values, 90)),
    ]
feature_config = FeatureExtractionConfig()

# for a quick training
if app_config.quick_limit is not None:
    train_paths, y_train = balanced_limit(train_paths, y_train, app_config.quick_limit)
    test_paths, y_test = balanced_limit(test_paths, y_test, app_config.quick_limit)

print(f"Train images: {len(train_paths)}")
print(f"Test images:  {len(test_paths)}")
print(f"Feature extraction config: {feature_config}")

X_train = build_feature_matrix(train_paths, feature_config, "train")
X_test = build_feature_matrix(test_paths, feature_config, "test")
 


# Modelling and validation
- **Model Choice & Pipeline**: `Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="rbf", class_weight="balanced"))])` — RBF SVM handles non-linear boundaries on moderate feature vectors; `class_weight="balanced"` mitigates class imbalance.

- **Hyperparameter Ranges**:
    - `svm__C = [0.1, 1, 10, 100]` — explores under/over-regularization regimes.
    - `svm__gamma = ["scale", 0.01, 0.001, 0.0001]` — includes heuristic and small numeric bandwidths; coarse grid reduces search cost for moderate datasets.

- **Cross-Validation & Training Strategy**:
    - `StratifiedKFold(n_splits = min(max_cv_folds, min_class_count), shuffle=True, random_state=42)` — preserves class proportions and avoids invalid fold counts.
    - `scoring="f1"` and `class_weight="balanced"` prioritize balanced precision/recall for defect detection.




In [ ]:
@dataclass(frozen=True)
class TrainingConfig:
    param_grid: dict[str, list[object]] = field(
        default_factory=lambda: {
            "svm__C": [0.1, 1, 10, 100],
            "svm__gamma": ["scale", 0.01, 0.001, 0.0001],
        }
    )
    max_cv_folds: int = 5
    scoring: str = "f1"
    random_state: int = 42
    # Parallel jobs for GridSearchCV. Use -1 outside restricted environments.
    n_jobs: int = 1
    verbose: int = 2


training_config = TrainingConfig()

# train model using SVM (RBF kernel)
def train_model(X_train: np.ndarray, y_train: np.ndarray, config: TrainingConfig) -> GridSearchCV:
    pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()), # StandardScaler standardizes features by removing the mean and scaling to unit variance, which is important for SVM performance since it relies on distances in feature space. This ensures that all features contribute equally to the model and prevents features with larger scales from dominating the decision boundary.
            ("svm", SVC(kernel="rbf", class_weight="balanced")), # The SVC with RBF kernel is a powerful non-linear classifier that can capture complex relationships in the data. The class_weight="balanced" option automatically adjusts weights inversely proportional to class frequencies, which helps address any class imbalance in the dataset and can improve performance on the minority class.
        ]
    )
    min_class_count = int(np.bincount(y_train).min())
    #at most configured max folds
    n_splits = min(config.max_cv_folds, min_class_count)
    if n_splits < 2:
        raise ValueError("Each class needs at least two training images.")
    #Stratified splitting keeps the same class proportions in every fold as in the whole dataset.
    #random_state ensures reproducibility, and shuffle=True randomizes the data before splitting to avoid any order bias.
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=config.random_state)
    #CV cross validation
    #GridSearchCV exhaustively tries all combinations of the specified hyperparameters (C and gamma for the SVM) and evaluates each combination using cross-validation on the training data. It selects the combination that yields the best average F1 score across the folds.
    search = GridSearchCV(
        pipeline,
        param_grid=config.param_grid,
        scoring=config.scoring,
        cv=cv,
        n_jobs=config.n_jobs,
        verbose=config.verbose,
    )
    search.fit(X_train, y_train)
    return search


print(f"Training feature matrix: {X_train.shape}")
print(f"Testing feature matrix:  {X_test.shape}")

model = train_model(X_train, y_train, training_config)

y_pred = model.predict(X_test)

report = classification_report(
    y_test,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
matrix = confusion_matrix(y_test, y_pred).tolist()
joblib.dump(model.best_estimator_, app_config.output_model)

print(f"Best parameters: {model.best_params_}")
print(f"Best CV F1: {model.best_score_:.4f}")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))
print("Confusion matrix:")
print(np.asarray(matrix))



# Explainable AI (XAI)
- **Occlusion Sensitivity Heatmaps**:
    - Occlusion sensitivity heatmaps (`occlusion_patch_size = 32`, `occlusion_stride = 16`) localize influential texture regions with manageable compute.
    - Occlusion tests the model’s response when local patches are masked, producing an importance map over the image.
    - It is useful for texture-driven models because it reveals whether the classifier depends on local texture regions instead of only global statistics.
    - Advantages: highlights regions that affect the decision, helps detect whether the model is using meaningful defect texture, and provides a sanity-check for trust and debugging.
    - Limitations: patch-based heatmaps are coarse, can be noisy for texture models, and may reflect correlated texture patterns rather than exact physical defect shapes.
    - Practical caveat: the attention may appear patchy and texture-driven, not aligned to consistent defect shapes, so use heatmaps as diagnostic guidance rather than precise localization.

#TODO interpret model using generated heatmaps 

In [ ]:


@dataclass(frozen=True)
class HeatmapConfig:
    heatmap_dir: Path = Path("steelblast_svm_glcm_dwt_focus_heatmaps")
    image_size: int = 256
    occlusion_patch_size: int = 32
    occlusion_stride: int = 16
    max_per_case: int | None = 3

heatmap_config = HeatmapConfig()
    
# Computes confidence score for each prediction
def predicted_label_confidences(model: Pipeline, features: np.ndarray) -> np.ndarray:
    predictions = model.predict(features)
    # decision_function gives distance to the separating hyperplane. For binary classification, it's a single score where positive means class 1 and negative means class 0. Magnitude of distance = confidence
    scores = model.decision_function(features)
    # 
    if scores.ndim == 1:
        positive_class = int(model.classes_[1])
        signed_scores = np.where(predictions == positive_class, scores, -scores)
        return signed_scores.astype(np.float64)

    return np.asarray(
        [scores[index, np.flatnonzero(model.classes_ == label)[0]] for index, label in enumerate(predictions)],
        dtype=np.float64,
    )

def prediction_confidence(model: Pipeline, features: np.ndarray, label: int) -> float:
    scores = model.decision_function(features.reshape(1, -1))

    if np.ndim(scores) == 1:
        positive_class = int(model.classes_[1])
        score = float(scores[0])
        return score if label == positive_class else -score

    class_index = int(np.flatnonzero(model.classes_ == label)[0])
    return float(scores[0, class_index])

def read_display_image(image_path: Path, image_size: int) -> np.ndarray:
    image = imread(image_path)

    if image.ndim == 2:
        image = np.repeat(image[..., np.newaxis], 3, axis=2)
    else:
        image = image[..., :3]

    image = resize(
        image,
        (image_size, image_size),
        anti_aliasing=True,
        preserve_range=True,
    )

    image = image.astype(np.float32)
    if image.max() > 1.0:
        image /= 255.0

    return np.clip(image, 0.0, 1.0)

def save_focus_pair(
    display_image: np.ndarray,
    heatmap: np.ndarray,
    title: str,
    output_path: Path,
) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), constrained_layout=True)
    axes[0].imshow(display_image)
    axes[0].set_title("Example image")
    axes[0].set_axis_off()

    axes[1].imshow(display_image)
    vmax = float(heatmap.max()) if heatmap.size and heatmap.max() > 0 else 1.0
    overlay = axes[1].imshow(heatmap, cmap="jet", alpha=0.62, vmin=0, vmax=vmax)
    axes[1].set_title("Activation heatmap")
    axes[1].set_axis_off()
    fig.colorbar(overlay, ax=axes[1], label="Decision-score drop")
    fig.suptitle(title, fontsize=16)
    fig.savefig(output_path, dpi=180)
    plt.close(fig)


# Mask parts of the image and see how prediction confidence changes.
# Steps:
# 1. Compute base prediction confidence
# 2. Slide a patch over the image
# 3. Replace patch with median value
# 4. Recompute prediction
# 5. Measure confidence drop
def compute_occlusion_focus_heatmap(
    image: np.ndarray,
    model: Pipeline,
    predicted_label: int,
    feature_config: FeatureExtractionConfig,
    patch_size: int, # The size of the square region (in pixels) that gets “hidden” at each step.
    stride: int, # The number of pixels the patch moves horizontally and vertically between steps. Smaller strides create smoother heatmaps but require more computations.
) -> np.ndarray:
    base_features = extract_features_from_image(image, feature_config)
    base_confidence = prediction_confidence(model, base_features, predicted_label)
    heatmap = np.zeros_like(image, dtype=np.float32)
    counts = np.zeros_like(image, dtype=np.float32)
    fill_value = float(np.median(image))
    height, width = image.shape

    for top in range(0, height, stride):
        bottom = min(top + patch_size, height)
        top = max(0, bottom - patch_size)

        for left in range(0, width, stride):
            right = min(left + patch_size, width)
            left = max(0, right - patch_size)

            occluded = image.copy()
            occluded[top:bottom, left:right] = fill_value
            occluded_features = extract_features_from_image(occluded, feature_config)
            occluded_confidence = prediction_confidence(model, occluded_features, predicted_label)
            importance = max(0.0, base_confidence - occluded_confidence)

            heatmap[top:bottom, left:right] += importance
            counts[top:bottom, left:right] += 1

    return np.divide(heatmap, counts, out=np.zeros_like(heatmap), where=counts > 0)

import shutil

def reset_directory(output_dir: Path):
    if output_dir.exists():
        shutil.rmtree(output_dir)  # ❌ delete everything inside
    output_dir.mkdir(parents=True, exist_ok=True)  # ✅ recreate empty folder

from pathlib import Path
import numpy as np
from typing import Dict, List, Any

def save_classification_case_heatmaps(
    test_paths: list[Path],
    y_test: np.ndarray,
    y_pred: np.ndarray,
    X_test: np.ndarray,
    model: Pipeline,
    feature_config: FeatureExtractionConfig,
    heatmap_config: HeatmapConfig,
) -> Dict[str, List[Dict[str, Any]]]:
    """
    Generate and save occlusion heatmaps for ALL classification cases (TP, FP, FN, TN).

    Returns:
        Dictionary mapping case -> list of metadata dicts.
    """

    reset_directory(heatmap_config.heatmap_dir)

    heatmap_results: Dict[str, List[Dict[str, Any]]] = {}
    confidences = predicted_label_confidences(model, X_test)

    case_definitions = {
        "true_positive": (1, 1),
        "false_positive": (0, 1),
        "false_negative": (1, 0),
        "true_negative": (0, 0),
    }

    for case_name, (actual_label, predicted_label) in case_definitions.items():

        # ✅ Collect indices
        case_indices = [
            i for i, (a, p) in enumerate(zip(y_test, y_pred))
            if a == actual_label and p == predicted_label
        ]

        # ✅ Sort by confidence (highest first)
        case_indices = sorted(
            case_indices,
            key=lambda i: confidences[i],
            reverse=True,
        )

        # ✅ Optional limit (useful for large datasets)
        if heatmap_config.max_per_case is not None:
            case_indices = case_indices[:heatmap_config.max_per_case]

        case_dir = heatmap_config.heatmap_dir / case_name
        case_dir.mkdir(parents=True, exist_ok=True)

        heatmap_results[case_name] = []

        print(f"\nProcessing {case_name}: {len(case_indices)} images")

        for rank, idx in enumerate(case_indices):

            try:
                image_path = test_paths[idx]

                # ✅ Load images
                grayscale_image = read_grayscale_image(
                    image_path, heatmap_config.image_size, feature_config
                )
                display_image = read_display_image(
                    image_path, heatmap_config.image_size
                )

                # ✅ Compute heatmap
                heatmap = compute_occlusion_focus_heatmap(
                    grayscale_image,
                    model,
                    int(y_pred[idx]),
                    feature_config,
                    heatmap_config.occlusion_patch_size,
                    heatmap_config.occlusion_stride,
                )

                # ✅ Create unique filename
                output_path = case_dir / (
                    f"{rank:03d}_{image_path.stem}_heatmap.png"
                )

                title = (
                    f"{case_name.replace('_', ' ').title()} | "
                    f"actual {CLASS_NAMES[int(y_test[idx])]}, "
                    f"predicted {CLASS_NAMES[int(y_pred[idx])]} | "
                    f"conf {confidences[idx]:.3f}"
                )

                # ✅ Save visualization
                save_focus_pair(display_image, heatmap, title, output_path)

                # ✅ Store metadata
                result = {
                    "index": int(idx),
                    "rank": rank,
                    "actual_label": CLASS_NAMES[int(y_test[idx])],
                    "predicted_label": CLASS_NAMES[int(y_pred[idx])],
                    "source_image": str(image_path),
                    "heatmap": str(output_path),
                    "confidence": float(confidences[idx]),
                    "max_activation": float(np.max(heatmap)),
                }

                heatmap_results[case_name].append(result)

            except Exception as e:
                print(f"⚠️ Failed on index {idx}: {e}")

        print(f"✅ Done {case_name}: saved {len(heatmap_results[case_name])}")

    return heatmap_results

heatmap_paths = save_classification_case_heatmaps(
    test_paths,
    y_test,
    y_pred,
    X_test,
    model.best_estimator_,
    feature_config,
    heatmap_config,
)


metrics = {
    "best_params": model.best_params_,
    "best_cv_f1": float(model.best_score_),
    "classification_report": report,
    "confusion_matrix": matrix,
    "train_images": len(train_paths),
    "test_images": len(test_paths),
    "feature_count": int(X_train.shape[1]),
    "feature_extraction_config": asdict(feature_config),
    "training_config": asdict(training_config),
    "heatmap_method": "occlusion_sensitivity",
    "heatmap_settings": {
        "max_per_case": heatmap_config.max_per_case,
        "image_size": heatmap_config.image_size,
        "occlusion_patch_size": heatmap_config.occlusion_patch_size,
        "occlusion_stride": heatmap_config.occlusion_stride,
    },
    "heatmaps": heatmap_paths,
}
app_config.metrics_json.write_text(json.dumps(metrics, indent=2))

print(f"Saved model to:   {app_config.output_model.resolve()}")
print(f"Saved metrics to: {app_config.metrics_json.resolve()}")
print(f"Saved heatmaps to: {heatmap_config.heatmap_dir.resolve()}")


# Bias and Diagnostic Analysis
- **Lighting Robustness Tests** assess whether the model’s predictions remain stable under realistic illumination perturbations.
- We apply controlled changes such as darkening, brightening, additive offset, and contrast shifts before the same CLAHE normalization and feature extraction pipeline.
- This diagnostic analysis checks for bias toward particular brightness or contrast conditions that could unfairly affect defect detection in industrial images.
- Advantages: it reveals whether model decisions depend on lighting artifacts rather than intrinsic texture features, and it quantifies prediction drift under operating variability.
- Limitations: these tests do not cover all real-world imaging variations (e.g. shadows, reflections, sensor noise), but they do provide a useful first check for illumination sensitivity.
- Practical interpretation: stable accuracy and low flip-rate under moderate lighting perturbations supports the claim that the model is more robust to illumination bias than a model trained without this normalization pipeline.
- Summary of results: the model passed the robustness criterion with min accuracy = 0.904 and max flip rate = 0.052 for darkening, additive offsets, and moderate contrast shifts.
- Claim: Model is robust to non-saturating lighting variation after CLAHE normalization.
- Criterion: For darkening, additive offsets, and moderate contrast shifts: accuracy >= 0.90 and flip_rate <= 0.06.
- Note: severe brightening is reported as a stress test because clipping can destroy texture information.

# Algorithmic Fairness
- In this defect inspection scenario, false positives and false negatives have distinct operational consequences.
- A false positive means a good part is flagged as defective, causing unnecessary rework, inspection cost, and lower manufacturing throughput.
- A false negative means a defective part is passed as good, which is more serious because it can lead to downstream failures, safety hazards, and customer dissatisfaction.
- For real-world fairness, the model should prioritize reducing false negatives while keeping false positives low enough to avoid excessive waste and mistrust in automated quality control.
- Confusion matrices, class-specific recall, and robustness checks under varied lighting help reveal whether the system is systematically biased toward one type of error.


In [ ]:
# Lighting robustness evaluation
# This tests whether the trained model keeps its predictions stable when test images are
# darkened, brightened, offset, or contrast-shifted before the same preprocessing pipeline.

def perturb_lighting(image: np.ndarray, perturbation: str) -> np.ndarray:
    if perturbation == "baseline":
        return image
    if perturbation == "dark_25":
        return np.clip(image * 0.75, 0.0, 1.0)
    if perturbation == "dark_50":
        return np.clip(image * 0.50, 0.0, 1.0)
    if perturbation == "bright_25":
        return np.clip(image * 1.25, 0.0, 1.0)
    if perturbation == "bright_50":
        return np.clip(image * 1.50, 0.0, 1.0)
    if perturbation == "offset_plus_10":
        return np.clip(image + 0.10, 0.0, 1.0)
    if perturbation == "offset_minus_10":
        return np.clip(image - 0.10, 0.0, 1.0)
    if perturbation == "low_contrast":
        return np.clip((image - 0.5) * 0.75 + 0.5, 0.0, 1.0)
    if perturbation == "high_contrast":
        return np.clip((image - 0.5) * 1.25 + 0.5, 0.0, 1.0)
    raise ValueError(f"Unknown lighting perturbation: {perturbation}")


def build_perturbed_feature_matrix(
    image_paths: list[Path],
    feature_config: FeatureExtractionConfig,
    perturbation: str,
) -> np.ndarray:
    features = []
    for image_path in image_paths:
        raw_image = read_grayscale_image(image_path, feature_config.image_size, config=None)
        perturbed_image = perturb_lighting(raw_image, perturbation)
        normalized_image = normalize_illumination(perturbed_image, feature_config)
        features.append(extract_features_from_image(normalized_image, feature_config))
    return np.vstack(features)


def evaluate_lighting_robustness(
    image_paths: list[Path],
    y_true: np.ndarray,
    baseline_predictions: np.ndarray,
    fitted_model: Pipeline,
    feature_config: FeatureExtractionConfig,
) -> list[dict[str, object]]:
    perturbations = [
        "baseline",
        "dark_25",
        "dark_50",
        "offset_plus_10",
        "offset_minus_10",
        "low_contrast",
        "high_contrast",
        "bright_25",
        "bright_50",
    ]
    normal_lighting_tests = {
        "dark_25",
        "dark_50",
        "offset_plus_10",
        "offset_minus_10",
        "low_contrast",
        "high_contrast",
    }
    rows = []

    for perturbation in perturbations:
        X_variant = build_perturbed_feature_matrix(
            image_paths,
            feature_config,
            perturbation,
        )
        y_variant = fitted_model.predict(X_variant)
        matrix_variant = confusion_matrix(y_true, y_variant, labels=[0, 1])
        accuracy = float(np.mean(y_variant == y_true))
        flips = int(np.sum(y_variant != baseline_predictions))
        good_recall = float(matrix_variant[0, 0] / matrix_variant[0].sum())
        not_good_recall = float(matrix_variant[1, 1] / matrix_variant[1].sum())
        rows.append(
            {
                "perturbation": perturbation,
                "accuracy": accuracy,
                "prediction_flips_vs_baseline": flips,
                "flip_rate_vs_baseline": float(flips / len(y_true)),
                "good_recall": good_recall,
                "not_good_recall": not_good_recall,
                "confusion_matrix": matrix_variant.tolist(),
                "included_in_robustness_claim": perturbation in normal_lighting_tests,
            }
        )

    return rows


lighting_robustness = evaluate_lighting_robustness(
    test_paths,
    y_test,
    y_pred,
    model.best_estimator_,
    feature_config,
)

print("Lighting robustness check")
print("perturbation       acc   flips  flip_rate  good_recall  not_good_recall")
for row in lighting_robustness:
    print(
        f"{row['perturbation']:16s} "
        f"{row['accuracy']:.3f} "
        f"{row['prediction_flips_vs_baseline']:5d} "
        f"{row['flip_rate_vs_baseline']:.3f} "
        f"{row['good_recall']:.3f} "
        f"{row['not_good_recall']:.3f}"
    )

claim_rows = [row for row in lighting_robustness if row["included_in_robustness_claim"]]
min_claim_accuracy = min(row["accuracy"] for row in claim_rows)
max_claim_flip_rate = max(row["flip_rate_vs_baseline"] for row in claim_rows)
lighting_robustness_summary = {
    "claim": "Model is robust to non-saturating lighting variation after CLAHE normalization.",
    "criterion": "For darkening, additive offsets, and moderate contrast shifts: accuracy >= 0.90 and flip_rate <= 0.06.",
    "passed": bool(min_claim_accuracy >= 0.90 and max_claim_flip_rate <= 0.06),
    "min_claim_accuracy": float(min_claim_accuracy),
    "max_claim_flip_rate": float(max_claim_flip_rate),
    "note": "Severe brightening is reported as a stress test because clipping can destroy texture information.",
}

print("\nRobustness summary")
print(json.dumps(lighting_robustness_summary, indent=2))

metrics_path = app_config.metrics_json
metrics_with_robustness = json.loads(metrics_path.read_text())
metrics_with_robustness["lighting_robustness"] = {
    "summary": lighting_robustness_summary,
    "results": lighting_robustness,
}
metrics_path.write_text(json.dumps(metrics_with_robustness, indent=2))
print(f"Saved lighting robustness results to: {metrics_path.resolve()}")


# Deployment Considerations

TODO add diagram and formating
🏗️ Architecture (Selected Design)
Containerized FastAPI service deployed on AWS SageMaker (real-time endpoint)

A Dockerized FastAPI API serves the model, accepting raw images as input.
The service bundles:

Preprocessing (GLCM + DWT feature extraction)
Normalization (StandardScaler)
SVC (RBF) inference


Deployed as a SageMaker real-time endpoint with CPU-based auto-scaling.

✅ Justification

FastAPI → low-latency, high-performance API suitable for real-time inference
Docker → ensures reproducibility and portability
SageMaker → managed deployment, scaling, and integration with monitoring
Bundled pipeline → prevents training–serving inconsistencies
CPU-only → cost-efficient for lightweight SVM inference


🧰 Tools & Libraries

Modeling & preprocessing: scikit-learn, joblib
API serving: FastAPI
Containerization: Docker
Model tracking/versioning: MLflow
Monitoring: Prometheus + Grafana (or SageMaker built-in tools)

✅ Justification

scikit-learn + joblib → simple, reliable pipeline serialization
FastAPI → modern, faster alternative to Flask
MLflow → experiment tracking and model registry support
Prometheus/Grafana → flexible, production-grade observability


📊 Observability & Monitoring

Track latency and throughput for system performance
Monitor prediction distribution and class-specific errors (FP/FN)
Implement drift detection:

Illumination drift (e.g., lighting condition changes)
Texture drift (e.g., manufacturing variations affecting GLCM/DWT features)


Log requests and predictions for debugging and quality control

✅ Justification

Ensures early detection of performance degradation
Identifies real-world data shifts affecting model reliability
Supports continuous quality assurance in industrial settings


🔐 Security

Validate and sanitize inputs (image format, resolution, value ranges)
Secure endpoints using API keys/OAuth + HTTPS/TLS
Apply rate limiting to prevent abuse and ensure stability


🔄 Lifecycle & Retraining

Collect labeled production data, especially failure cases
Retrain periodically or when performance metrics degrade
Use MLflow model versioning to manage updates
Deploy new models using A/B testing or shadow deployment before full rollout

✅ Justification

Keeps the model aligned with evolving production conditions
Reduces risk when deploying updates
Enables reproducibility and controlled experimentation
